# Notebook 1: Tokenization
**LLM Fundamentals Demo Series — Agentic AI Bootcamp**

This notebook demonstrates how text is converted into tokens before being fed into an LLM.
We use **GPT-2** (117M params, fully public) and **DistilBERT** as our small reference models.

Topics covered:
1. What a token is — BPE (Byte-Pair Encoding)
2. Colorized token visualization
3. Vocabulary size and token IDs
4. Why models struggle with character-level tasks (the 'Strawberry' problem)
5. Token cost comparison across different text types

In [ ]:
# Install dependencies (run once)
!pip install transformers tiktoken matplotlib seaborn --quiet

In [ ]:
import re
from transformers import GPT2Tokenizer, AutoTokenizer
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import numpy as np

# Load the GPT-2 tokenizer (downloads ~0.5 MB)
gpt2_tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
bert_tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')

print(f'GPT-2 vocabulary size : {gpt2_tokenizer.vocab_size:,}')
print(f'DistilBERT vocabulary size: {bert_tokenizer.vocab_size:,}')

## 1. Basic Tokenization — What Does a Token Look Like?
GPT-2 uses **Byte-Pair Encoding (BPE)**: common character sequences are merged into single tokens.
The `Ġ` character (displayed as `▁`) represents a leading space.

In [ ]:
sentence = "The transformer architecture revolutionized natural language processing."

token_ids = gpt2_tokenizer.encode(sentence)
tokens    = [gpt2_tokenizer.decode([t]) for t in token_ids]

print(f'Input text  : "{sentence}"')
print(f'Token count : {len(token_ids)}')
print()
print(f'{"Token":<25} {"ID":>6}')
print('-' * 33)
for tok, tid in zip(tokens, token_ids):
    print(f'{repr(tok):<25} {tid:>6}')

## 2. Colorized Token Visualization
Each color represents a different token — notice how subword splitting works.

In [ ]:
def visualize_tokens(text, tokenizer, title='Token Visualization'):
    """Display text with each token highlighted in a distinct color."""
    token_ids = tokenizer.encode(text)
    tokens    = [tokenizer.decode([t]) for t in token_ids]

    # Build a color palette
    palette = plt.cm.get_cmap('tab20', len(tokens))
    colors  = [palette(i) for i in range(len(tokens))]

    fig, ax = plt.subplots(figsize=(14, 1.8))
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')
    ax.set_title(f'{title}  ({len(tokens)} tokens)', fontsize=13, pad=10)

    x = 0.01
    for tok, color in zip(tokens, colors):
        display = tok.replace(' ', '▁')  # show spaces visually
        txt = ax.text(x, 0.45, display,
                      fontsize=11, fontfamily='monospace',
                      bbox=dict(facecolor=color, alpha=0.6, boxstyle='round,pad=0.3',
                                edgecolor='grey', linewidth=0.5))
        fig.canvas.draw()
        bb = txt.get_window_extent(renderer=fig.canvas.get_renderer())
        width_norm = (bb.width + 10) / fig.get_size_inches()[0] / fig.dpi
        x += width_norm
        if x > 0.98:
            break  # avoid overflow

    plt.tight_layout()
    plt.show()

# --- Demo sentences ---
examples = [
    "The quick brown fox jumps over the lazy dog.",
    "Transformers use self-attention mechanisms.",
    "Tokenization splits text into subword units.",
]

for ex in examples:
    visualize_tokens(ex, gpt2_tokenizer, title=f'GPT-2')

## 3. The 'Strawberry' Problem — Why LLMs Struggle with Spelling
Models don't see individual characters; they see token IDs. Letter-counting tasks require the model to reason about the *internal structure* of tokens it never directly observes.

In [ ]:
def token_breakdown(word, tokenizer, label=''):
    ids    = tokenizer.encode(word, add_special_tokens=False)
    tokens = [tokenizer.decode([i]) for i in ids]
    chars  = list(word)
    r_count = word.lower().count('r')
    print(f'  Word     : "{word}"')
    print(f'  Tokens   : {tokens}  (count: {len(ids)})')
    print(f'  IDs      : {ids}')
    print(f'  Actual r count in word: {r_count}')
    print()

print('=== GPT-2 Tokenization of tricky words ===')
for word in ['strawberry', 'Strawberry', 'STRAWBERRY', 'banana', 'acknowledgement', '🍓']:
    token_breakdown(word, gpt2_tokenizer)

## 4. Token Count Comparison — Different Languages & Code
Token efficiency varies by language and content type, which directly affects **API cost** (you pay per token).

In [ ]:
samples = {
    'English prose': 'The model predicts the next token using a softmax over the vocabulary.',
    'Python code':   'def add(a, b):\n    return a + b\nresult = add(3, 4)',
    'JSON':          '{"name": "Alice", "role": "analyst", "score": 98.5}',
    'Spanish':       'El modelo predice el siguiente token usando una función softmax.',
    'Chinese':       '模型使用softmax函数预测下一个词元。',
    'Emoji':         '🔥🚀💡🤖🌍🎯🧠⚡🌊🎉',
    'Long number':   '3.14159265358979323846264338327950288419716939937510',
}

labels, gpt2_counts, bert_counts = [], [], []
for label, text in samples.items():
    g_ids = gpt2_tokenizer.encode(text)
    b_ids = bert_tokenizer.encode(text, add_special_tokens=False)
    labels.append(label)
    gpt2_counts.append(len(g_ids))
    bert_counts.append(len(b_ids))
    print(f'{label:<20}  GPT-2: {len(g_ids):3d} tokens  |  DistilBERT: {len(b_ids):3d} tokens')

# Bar chart comparison
x = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 5))
bars1 = ax.bar(x - width/2, gpt2_counts, width, label='GPT-2', color='steelblue', alpha=0.85)
bars2 = ax.bar(x + width/2, bert_counts,  width, label='DistilBERT', color='coral', alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=25, ha='right', fontsize=10)
ax.set_ylabel('Number of tokens')
ax.set_title('Token Count by Text Type: GPT-2 vs DistilBERT', fontsize=13)
ax.legend()
ax.bar_label(bars1, padding=2, fontsize=9)
ax.bar_label(bars2, padding=2, fontsize=9)
plt.tight_layout()
plt.show()

## 5. Special Tokens — [CLS], [SEP], [MASK]
BERT-family models add special tokens to mark the start/end of sequences and enable masked language modeling.

In [ ]:
text = "Attention is all you need."

encoding = bert_tokenizer(text, return_tensors='pt')
ids      = encoding['input_ids'][0].tolist()
tokens   = bert_tokenizer.convert_ids_to_tokens(ids)

print('DistilBERT tokenization with special tokens:')
print(f'{"Token":<15} {"ID":>6}')
print('-' * 23)
for tok, tid in zip(tokens, ids):
    marker = ' <-- special' if tok in ['[CLS]', '[SEP]', '[PAD]', '[MASK]', '[UNK]'] else ''
    print(f'{tok:<15} {tid:>6}{marker}')

print(f'\nTotal tokens (including specials): {len(ids)}')

## 6. BPE Vocabulary Distribution
Visualize how GPT-2's vocabulary is distributed across token lengths.

In [ ]:
vocab = gpt2_tokenizer.get_vocab()  # dict of token_str -> id

# Decode every token to get its string and length
lengths = []
for token_str in vocab.keys():
    # decode the bytes back to readable text (strip leading Ġ)
    decoded = token_str.replace('Ġ', ' ').replace('Ċ', '\n')
    lengths.append(len(decoded.strip()))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Histogram of token lengths
ax = axes[0]
ax.hist(lengths, bins=range(0, 20), color='steelblue', edgecolor='white', alpha=0.85)
ax.set_xlabel('Token length (characters)')
ax.set_ylabel('Count')
ax.set_title(f'GPT-2 Vocabulary: Token Length Distribution\n(total vocab = {len(vocab):,})', fontsize=11)
ax.set_xticks(range(0, 20))

# Pie chart of single-char vs multi-char tokens
ax2 = axes[1]
single = sum(1 for l in lengths if l == 1)
multi  = len(lengths) - single
ax2.pie([single, multi],
        labels=[f'Single char\n({single:,})', f'Multi char\n({multi:,})'],
        colors=['steelblue', 'coral'],
        autopct='%1.1f%%', startangle=90)
ax2.set_title('Single vs Multi-character Tokens', fontsize=11)

plt.suptitle('GPT-2 Vocabulary Analysis', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## Summary
- Tokenizers convert raw text into integer IDs via learned sub-word vocabularies (BPE, WordPiece).
- Token boundaries ≠ word boundaries → letter-counting, arithmetic, and spelling tasks are hard.
- Longer tokens are more efficient; non-Latin scripts and emoji use more tokens per character.
- BERT adds `[CLS]` and `[SEP]` special tokens; GPT-2 uses byte-level BPE without sentence-level specials.

> **Next notebook:** Embeddings & Vector Space — see how tokens become dense semantic vectors.